# Análise de Aderência de Candidatos — Setor Bancário

Este notebook analisa os dados reais (anonimizados) de 90 candidatos que responderam ao formulário de mapeamento de perfil em 2020.

O objetivo é entender a distribuição dos perfis, calibrar os pesos do scoring e identificar os padrões que diferenciam candidatos de alto e baixo alinhamento.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'

df = pd.read_csv('../data/sample_responses.csv')
print(f"Dataset: {len(df)} candidatos | {df.columns.tolist()}")
df.head()

## 1. Distribuição do Score de Aderência

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
axes[0].hist(df['score_aderencia'], bins=20, color='#2563eb', alpha=0.8, edgecolor='white')
axes[0].axvline(0.70, color='#16a34a', linestyle='--', linewidth=2, label='Alto Alinhamento (≥70%)')
axes[0].axvline(0.50, color='#d97706', linestyle='--', linewidth=2, label='Alinhamento Médio (≥50%)')
axes[0].set_title('Distribuição do Score de Aderência', fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Nº de Candidatos')
axes[0].legend(fontsize=9)

# Classificação
def classify(s):
    if s >= 0.70: return 'Alto Alinhamento'
    elif s >= 0.50: return 'Alinhamento Médio'
    elif s >= 0.30: return 'Alinhamento Baixo'
    else: return 'Fora do Perfil'

df['classificacao'] = df['score_aderencia'].apply(classify)
counts = df['classificacao'].value_counts()
colors = ['#16a34a','#d97706','#dc2626','#64748b']
axes[1].pie(counts, labels=counts.index, autopct='%1.0f%%', colors=colors, startangle=90)
axes[1].set_title('Distribuição por Classificação', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nEstatísticas do Score:")
print(df['score_aderencia'].describe().round(3))

## 2. Impacto de Cada Pergunta no Score Final

In [ ]:
perguntas = {
    'q1_idade': 'Idade',
    'q2_perfil': 'Perfil Comportamental',
    'q3_oportunidade': 'Oportunidade Desejada',
    'q4_certificacao': 'Certificação ANBIMA'
}

# Shortening long labels
short = {
    'Gosto de falar muito, sou comunicativo e, se deixar, converso o dia todo': 'Comunicativo',
    'Sou orientado a execução, gosto de fazer o que me pedem sem questionar': 'Executor',
    'Sou ambicioso, sei onde quero chegar e, se não atingir meus objetivos em determinado espaço de tempo, procuro outra oportunidade': 'Ambicioso',
    'Tenho uma boa capacidade analítica e conhecimento de ferramentas de informática': 'Analítico',
    'Sou orientado a inovação, questiono construtivamente os processos que julgo desalinhados': 'Inovador',
    'Atendimento ao público / Posso Ajudar?': 'Atendimento',
    'Caixa / Área interna': 'Caixa / Interno',
    'Estratégias, definição e acompanhamento de metas': 'Estratégia',
    'Análise de informações e identificação de oportunidades através da construção de materiais como planilhas em Excell e apresentações em Powerpoint': 'Análise/BI',
}
for col in ['q2_perfil', 'q3_oportunidade']:
    df[col] = df[col].replace(short)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(perguntas.items()):
    grp = df.groupby(col)['score_aderencia'].mean().sort_values(ascending=True)
    bars = axes[i].barh(grp.index, grp.values, color='#2563eb', alpha=0.85)
    axes[i].axvline(df['score_aderencia'].mean(), color='#dc2626', linestyle='--', linewidth=1.5, label='Média geral')
    axes[i].set_title(label, fontweight='bold')
    axes[i].set_xlabel('Score médio de aderência')
    axes[i].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    axes[i].legend(fontsize=8)
    for bar, val in zip(bars, grp.values):
        axes[i].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                     f'{val:.0%}', va='center', fontsize=9)

plt.suptitle('Score médio por resposta — identificando o que mais impacta a aderência',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Volume de Respostas ao Longo do Tempo

In [ ]:
df['data_resposta'] = pd.to_datetime(df['data_resposta'])
daily = df.groupby('data_resposta').size().reset_index(name='respostas')

plt.figure(figsize=(12, 4))
plt.bar(daily['data_resposta'], daily['respostas'], color='#2563eb', alpha=0.8, width=1)
plt.title('Volume de Respostas por Dia', fontweight='bold')
plt.xlabel('Data')
plt.ylabel('Nº de Respostas')
plt.tight_layout()
plt.show()

print(f"Período: {df['data_resposta'].min().date()} → {df['data_resposta'].max().date()}")
print(f"Pico diário: {daily['respostas'].max()} respostas em {daily.loc[daily['respostas'].idxmax(),'data_resposta'].date()}")

## 4. Calibração dos Pesos

Os pesos foram definidos a partir do range de variação que cada pergunta provoca no score final.
Uma pergunta com maior range tem maior poder discriminatório — e deve receber mais peso.

In [ ]:
ranges = {}
for col, label in perguntas.items():
    grp = df.groupby(col)['score_aderencia'].mean()
    ranges[label] = grp.max() - grp.min()

total = sum(ranges.values())
pesos = {k: v/total for k, v in ranges.items()}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(pesos.keys()), list(pesos.values()), color='#2563eb', alpha=0.85)
for bar, val in zip(bars, pesos.values()):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.0%}', va='center', fontweight='bold')
ax.set_title('Peso relativo de cada pergunta no score final', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
plt.tight_layout()
plt.show()

print("\nPesos sugeridos:")
for k, v in pesos.items():
    print(f"  {k}: {v:.2%}")